# Plumbing Agents: balancing availability, aging stock, and cash

A plumbing supplier needs common parts on the shelf, but every item held too long consumes cash that could fund payroll, deliveries, or faster-moving products. This lab builds a small multi-agent workflow around that tension.

The code is deliberately deterministic and inspectable. It uses **MCP-style tools** and **A2A-style messages** in-process so we can examine the architecture before replacing these teaching adapters with protocol-compliant services. No model API key is required.

## The working agreement

Three agents have narrow responsibilities:

- **Inventory Agent:** forecasts stock needs and flags aging inventory.
- **Accounting Agent:** owns private finance data and shares only purchase guardrails.
- **Purchasing Agent:** chooses proposed orders that fit those guardrails.

Prompts describe desired behavior. MCP authorization controls which deterministic tools an agent may call. A2A contracts control which fields can cross between agents. That distinction matters: confidentiality is enforced by the plumbing, not entrusted to an agent's discretion.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from math import ceil
from pathlib import Path
from typing import Any, Callable
from uuid import uuid4

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

LOCAL_DATA = Path('../sample_data')
REMOTE_DATA = 'https://raw.githubusercontent.com/ChrisHartline/plumbingagents/main/sample_data'
DATA_SOURCE = LOCAL_DATA if LOCAL_DATA.exists() else REMOTE_DATA

def load_csv(filename: str) -> pd.DataFrame:
    source = DATA_SOURCE / filename if isinstance(DATA_SOURCE, Path) else f'{DATA_SOURCE}/{filename}'
    return pd.read_csv(source)

inventory = load_csv('inventory.csv')
sales = load_csv('sales_history.csv')
suppliers = load_csv('supplier_terms.csv')
cash_constraints = load_csv('cash_constraints.csv')

display(inventory)

## Agent instructions

These short instructions are ready to become system prompts when language models are added. The first lab keeps the decisions deterministic, which makes every number reproducible and every permission testable.

In [ ]:
SYSTEM_PROMPTS = {
    'inventory_agent': (
        'Protect product availability while identifying cash trapped in aging stock. '
        'Use approved inventory, demand, and supplier tools. Do not request accounting records.'
    ),
    'accounting_agent': (
        'Protect confidential financial data. Convert private records into the smallest '
        'purchase guardrails needed by another agent.'
    ),
    'purchasing_agent': (
        'Build an explainable purchase plan from inventory proposals and disclosed cash '
        'guardrails. Never infer or request undisclosed balances.'
    ),
}

pd.Series(SYSTEM_PROMPTS, name='system_prompt').to_frame()

## A transparent forecasting and aging model

For the first experiment, forecast demand is the eight-week average. Target stock covers supplier lead time plus one review week, then adds safety stock. Order quantities respect supplier case packs and minimums. This is a baseline we can later compare against seasonality-aware or probabilistic ML models.

In [ ]:
def rounded_order_quantity(shortage: int, case_pack: int, minimum_order_qty: int) -> int:
    if shortage <= 0:
        return 0
    packed_quantity = ceil(shortage / case_pack) * case_pack
    return max(minimum_order_qty, packed_quantity)


def forecast_stock_needs(
    inventory_df: pd.DataFrame,
    sales_df: pd.DataFrame,
    supplier_df: pd.DataFrame,
    review_period_weeks: int = 1,
) -> pd.DataFrame:
    demand = (
        sales_df.groupby('sku', as_index=False)['units_sold']
        .mean()
        .rename(columns={'units_sold': 'forecast_weekly_demand'})
    )
    result = inventory_df.merge(demand, on='sku').merge(supplier_df, on='sku')
    result['target_stock'] = result.apply(
        lambda row: ceil(
            row['forecast_weekly_demand'] * (row['lead_time_weeks'] + review_period_weeks)
            + row['safety_stock']
        ),
        axis=1,
    )
    result['shortage'] = (result['target_stock'] - result['on_hand']).clip(lower=0).astype(int)
    result['order_qty'] = result.apply(
        lambda row: rounded_order_quantity(
            row['shortage'], int(row['case_pack']), int(row['minimum_order_qty'])
        ),
        axis=1,
    )
    result['estimated_cost'] = (result['order_qty'] * result['unit_price']).round(2)
    result['weeks_of_cover'] = (
        result['on_hand'] / result['forecast_weekly_demand'].replace(0, pd.NA)
    ).round(1)
    result['priority_score'] = (
        100 * result['shortage'] / result['target_stock'].clip(lower=1)
        + 5 * result['lead_time_weeks']
    ).round(1)
    result['aging_risk'] = (
        (result['oldest_age_days'] >= 180) & (result['weeks_of_cover'] >= 8)
    )
    result['aging_action'] = result['aging_risk'].map(
        {True: 'Freeze reorder; review transfer, return, or markdown', False: 'No aging action'}
    )
    return result


model_preview = forecast_stock_needs(inventory, sales, suppliers)
display(model_preview[[
    'sku', 'description', 'forecast_weekly_demand', 'on_hand', 'target_stock',
    'order_qty', 'estimated_cost', 'weeks_of_cover', 'aging_risk'
]])

## MCP-style tools: capability and data boundaries

Each tool declares its allowed callers and data classification. The gateway checks the caller before invoking deterministic code and writes an audit event. In production, these handlers would sit behind real MCP servers and identity-aware authorization.

In [ ]:
@dataclass(frozen=True)
class ToolSpec:
    name: str
    allowed_agents: frozenset[str]
    classification: str
    handler: Callable[..., Any]


class MCPGateway:
    def __init__(self) -> None:
        self.tools: dict[str, ToolSpec] = {}
        self.audit_log: list[dict[str, Any]] = []

    def register(self, spec: ToolSpec) -> None:
        self.tools[spec.name] = spec

    def call(self, caller: str, tool_name: str, **kwargs: Any) -> Any:
        spec = self.tools[tool_name]
        allowed = caller in spec.allowed_agents
        self.audit_log.append({
            'caller': caller,
            'tool': tool_name,
            'classification': spec.classification,
            'allowed': allowed,
        })
        if not allowed:
            raise PermissionError(f'{caller} may not call {tool_name}')
        return spec.handler(**kwargs)


def inventory_positions() -> pd.DataFrame:
    return inventory.copy()


def demand_history() -> pd.DataFrame:
    return sales.copy()


def supplier_catalog() -> pd.DataFrame:
    return suppliers.copy()


def private_purchase_guardrails() -> dict[str, Any]:
    row = cash_constraints.iloc[0]
    return {
        'period': row['period'],
        'purchasing_budget': float(row['purchasing_budget']),
        'minimum_cash_reserve': float(row['minimum_cash_reserve']),
        'max_single_po': float(row['max_single_po']),
        'data_owner': row['data_owner'],
        'sharing_policy': row['sharing_policy'],
    }


mcp = MCPGateway()
mcp.register(ToolSpec('inventory.positions', frozenset({'inventory_agent'}), 'internal', inventory_positions))
mcp.register(ToolSpec('sales.demand_history', frozenset({'inventory_agent'}), 'internal', demand_history))
mcp.register(ToolSpec('supplier.catalog', frozenset({'inventory_agent'}), 'internal', supplier_catalog))
mcp.register(ToolSpec(
    'finance.purchase_guardrails',
    frozenset({'accounting_agent'}),
    'confidential',
    private_purchase_guardrails,
))

## A2A-style messages: explicit contracts between agents

The gateway permits only known sender-recipient-action combinations. It also applies a field allowlist to every payload. The Accounting Agent can read confidential guardrails, but only four fields are permitted in its message to Purchasing.

In [ ]:
@dataclass
class A2AMessage:
    message_id: str
    task_id: str
    sender: str
    recipient: str
    action: str
    payload: dict[str, Any]
    data_classification: str
    removed_fields: list[str]
    schema_version: str = '0.1'


class A2AGateway:
    def __init__(self) -> None:
        self.contracts: dict[tuple[str, str, str], dict[str, Any]] = {
            ('inventory_agent', 'purchasing_agent', 'inventory.stock_assessment'): {
                'fields': {'as_of', 'reorder_proposals', 'aging_watchlist'},
                'classification': 'internal',
            },
            ('accounting_agent', 'purchasing_agent', 'finance.purchase_guardrails'): {
                'fields': {'period', 'purchasing_budget', 'minimum_cash_reserve', 'max_single_po'},
                'classification': 'restricted-summary',
            },
        }
        self.audit_log: list[dict[str, Any]] = []

    def send(
        self,
        task_id: str,
        sender: str,
        recipient: str,
        action: str,
        payload: dict[str, Any],
    ) -> A2AMessage:
        key = (sender, recipient, action)
        if key not in self.contracts:
            raise PermissionError(f'No A2A contract for {key}')
        contract = self.contracts[key]
        allowed_fields = contract['fields']
        clean_payload = {key: value for key, value in payload.items() if key in allowed_fields}
        removed_fields = sorted(set(payload) - allowed_fields)
        message = A2AMessage(
            message_id=str(uuid4()),
            task_id=task_id,
            sender=sender,
            recipient=recipient,
            action=action,
            payload=clean_payload,
            data_classification=contract['classification'],
            removed_fields=removed_fields,
        )
        self.audit_log.append({
            'message_id': message.message_id,
            'sender': sender,
            'recipient': recipient,
            'action': action,
            'classification': message.data_classification,
            'removed_fields': removed_fields,
        })
        return message


a2a = A2AGateway()

## The agents

The agents are small coordinators. They do not contain database access or secret-handling logic; they can only use the gateways they receive. This keeps business reasoning separate from transport and authorization.

In [ ]:
class InventoryAgent:
    name = 'inventory_agent'

    def assess(self, task_id: str) -> A2AMessage:
        positions = mcp.call(self.name, 'inventory.positions')
        history = mcp.call(self.name, 'sales.demand_history')
        terms = mcp.call(self.name, 'supplier.catalog')
        assessment = forecast_stock_needs(positions, history, terms)

        reorder_columns = [
            'sku', 'description', 'supplier', 'order_qty', 'unit_price',
            'estimated_cost', 'priority_score', 'target_stock', 'on_hand',
        ]
        aging_columns = [
            'sku', 'description', 'on_hand', 'oldest_age_days',
            'weeks_of_cover', 'aging_action',
        ]
        payload = {
            'as_of': cash_constraints.iloc[0]['period'],
            'reorder_proposals': assessment.loc[assessment['order_qty'] > 0, reorder_columns].to_dict('records'),
            'aging_watchlist': assessment.loc[assessment['aging_risk'], aging_columns].to_dict('records'),
        }
        return a2a.send(
            task_id, self.name, 'purchasing_agent', 'inventory.stock_assessment', payload
        )


class AccountingAgent:
    name = 'accounting_agent'

    def share_guardrails(self, task_id: str) -> A2AMessage:
        private_summary = mcp.call(self.name, 'finance.purchase_guardrails')
        return a2a.send(
            task_id, self.name, 'purchasing_agent', 'finance.purchase_guardrails', private_summary
        )


class PurchasingAgent:
    name = 'purchasing_agent'

    def plan(
        self, inventory_message: A2AMessage, finance_message: A2AMessage
    ) -> dict[str, Any]:
        candidates = sorted(
            inventory_message.payload['reorder_proposals'],
            key=lambda item: (-item['priority_score'], item['sku']),
        )
        budget = finance_message.payload['purchasing_budget']
        max_single_po = finance_message.payload['max_single_po']
        selected: list[dict[str, Any]] = []
        deferred: list[dict[str, Any]] = []
        committed = 0.0

        for candidate in candidates:
            cost = float(candidate['estimated_cost'])
            if cost > max_single_po:
                deferred.append({**candidate, 'reason': 'exceeds per-order limit'})
            elif committed + cost > budget:
                deferred.append({**candidate, 'reason': 'outside current purchase budget'})
            else:
                selected.append(candidate)
                committed = round(committed + cost, 2)

        return {
            'period': finance_message.payload['period'],
            'budget': budget,
            'committed': committed,
            'remaining': round(budget - committed, 2),
            'selected': selected,
            'deferred': deferred,
            'aging_watchlist': inventory_message.payload['aging_watchlist'],
        }

## Run one purchasing cycle

The two specialist agents send independently governed messages. Purchasing combines those messages without receiving the underlying accounting table.

In [ ]:
task_id = 'purchase-review-2026-W33'
inventory_message = InventoryAgent().assess(task_id)
finance_message = AccountingAgent().share_guardrails(task_id)
purchase_plan = PurchasingAgent().plan(inventory_message, finance_message)

print(
    f"Budget: ${purchase_plan['budget']:,.2f} | "
    f"Committed: ${purchase_plan['committed']:,.2f} | "
    f"Remaining: ${purchase_plan['remaining']:,.2f}"
)

print('\nSelected purchase lines')
display(pd.DataFrame(purchase_plan['selected'])[[
    'sku', 'description', 'supplier', 'order_qty', 'estimated_cost', 'priority_score'
]])

print('\nDeferred purchase lines')
display(pd.DataFrame(purchase_plan['deferred'])[[
    'sku', 'description', 'order_qty', 'estimated_cost', 'priority_score', 'reason'
]])

print('\nAging watchlist')
display(pd.DataFrame(purchase_plan['aging_watchlist']))

## Inspect the confidentiality controls

The first check shows field-level filtering at the A2A boundary. The second proves that Purchasing cannot bypass Accounting and call the confidential finance tool directly.

In [ ]:
print('Fields received by Purchasing:', sorted(finance_message.payload))
print('Fields removed by A2A policy:', finance_message.removed_fields)

try:
    mcp.call('purchasing_agent', 'finance.purchase_guardrails')
except PermissionError as error:
    print('Blocked direct access:', error)

print('\nMCP audit trail')
display(pd.DataFrame(mcp.audit_log))

print('\nA2A audit trail')
display(pd.DataFrame(a2a.audit_log))

## What this gives us

The Inventory Agent is a useful grouping: forecasting and aging share the same inventory context and produce one stock-health assessment. Accounting remains separate because it owns a different trust boundary. Purchasing receives enough information to act, but not enough to reconstruct the company's cash position.

The next experiment can replace the moving-average baseline with a seasonality-aware model, add manager approval for exceptions, and extract the MCP gateway, A2A contracts, and agents into tested package modules. The notebook remains the readable narrative; the package becomes the reusable engineering artifact.